In [1]:
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
import numpy as np
import pickle

In [2]:
device ="mps"

In [38]:
with open('../../data/complete_features.pkl', 'rb') as f:
    all_data = pickle.load(f)

In [39]:
data = all_data["scaled_featured"].copy()
data["close"] = all_data["Close"]

In [40]:
# log returns
data["log_return"] = np.log(data["close"] / data["close"].shift(1))
data.dropna(inplace=True)

In [41]:
data

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20,close,log_return
Date,,,,,,,,,,,,,,
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460,3988.550049,-0.010339
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134,4157.100098,0.041390
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557,4162.200195,0.001226
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752,4049.000000,-0.027574
2008-07-14,-1.257924,-1.882225,2.371776,-0.148393,-1.268721,-0.518601,-0.436624,-3.173512,1.521334,0.430252,-2.071028,-0.455743,4039.699951,-0.002300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783,25069.199219,-0.001785
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354,25239.099609,0.006754
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624,25330.250000,0.003605


In [42]:
close = data["close"]
log_return = data["log_return"]

In [43]:
data.shape

(4218, 14)

In [44]:
data.drop(columns=["close","log_return"], inplace=True)

In [45]:
data

,rolling_sharpe_20,rolling_max_dd_60,realized_vol_20,vol_slope_20,autocorr_20,rel_spx_20,usdinr_vol_20,rel_gold_20,vix_level_20,vix_slope_20,trend_strength_200,sector_dispersion_ratio_20
Date,,,,,,,,,,,,
2008-07-08,-1.211754,-1.882225,2.077736,-0.022097,-1.117078,-1.046961,-0.542321,-2.455523,1.450259,0.981827,-2.206775,-0.513460
2008-07-09,-0.957426,-1.882225,2.289506,1.636228,-1.218247,-0.338846,-0.561373,-1.988903,1.476862,0.785786,-1.960992,-0.565134
2008-07-10,-0.975703,-1.882225,2.287995,-0.006124,-1.013921,-0.474801,-0.596656,-2.401313,1.493754,0.505013,-1.940271,-0.562557
2008-07-11,-1.136144,-1.882225,2.391761,0.804554,-1.050675,-0.396400,-0.475448,-2.987661,1.507027,0.400347,-2.078101,-0.510752
2008-07-14,-1.257924,-1.882225,2.371776,-0.148393,-1.268721,-0.518601,-0.436624,-3.173512,1.521334,0.430252,-2.071028,-0.455743
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-15,0.375855,0.654743,-0.795604,0.021503,0.541634,-0.126723,-0.727148,-1.189892,-0.988672,-0.309058,0.314003,0.507783
2025-09-16,0.280399,0.654743,-0.833663,-0.287568,0.045456,-0.238463,-0.708679,-1.276291,-1.000565,-0.327333,0.453848,0.513354
2025-09-17,0.260114,0.654743,-0.836851,-0.019044,0.154682,-0.363408,-0.703390,-1.335486,-1.009414,-0.239281,0.522324,0.511624


In [32]:
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, data, window_size=60):
        """
        data: numpy array [T, D]
        """
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size + 1

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.window_size]
        return x

window_size = 60
val_size = 0

train_size = int(len(data) * (1 - val_size))
train_data = data.iloc[:train_size]
val_data = data.iloc[train_size:]

train_dataset = TimeSeriesWindowDataset(train_data.values, window_size)
val_dataset = TimeSeriesWindowDataset(val_data.values, window_size)
combined_dataset = TimeSeriesWindowDataset(data.values, window_size)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
combined_loader = DataLoader(combined_dataset, batch_size=batch_size, shuffle=False)



In [33]:
import torch.nn as nn
import torch.nn.functional as F

class TS2VecModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=5):
        super(TS2VecModel, self).__init__()
        layers = []
        for i in range(num_layers):
            dilation = 2 ** i
            layers.append(nn.Conv1d(input_dim if i == 0 else hidden_dim, hidden_dim, kernel_size=3, padding=dilation, dilation=dilation))
            layers.append(nn.ReLU())
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        # x: [B, T, D]
        x = x.permute(0, 2, 1)  # [B, D, T]
        x = self.network(x)     # [B, H, T]
        x = x.permute(0, 2, 1)  # [B, T, H]
        # normalize
        x = F.normalize(x, p=2, dim=-1)
        return x
    
model = TS2VecModel(input_dim=data.shape[1], hidden_dim=128, num_layers=6).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [34]:
def time_mask(x, mask_ratio=0.2):
    B, T, D = x.shape
    mask_len = int(T * mask_ratio)
    start = np.random.randint(0, T - mask_len)
    x = x.clone()
    x[:, start:start+mask_len, :] = 0
    return x


def jitter(x, sigma=0.02):
    return x + sigma * torch.randn_like(x)


def temporal_pooling(z):
    if z.size(1) % 2 == 1:
        z = z[:, :-1]
    return z.reshape(z.size(0), z.size(1)//2, 2, z.size(2)).mean(dim=2)


def ts2vec_contrastive_loss_vectorized(
    z1, z2,
    temperature=0.1,
    base_exclusion_radius=5
):
    """
    z1, z2: [B, T, C]
    """
    B, T, C = z1.shape
    device = z1.device

    # flatten (instance, time)
    z1_flat = z1.reshape(B*T, C)
    z2_flat = z2.reshape(B*T, C)

    # cosine similarity == dot product because embeddings are normalized
    sim = torch.matmul(z1_flat, z2_flat.T) / temperature   # [BT, BT]

    # ----- temporal negative mask -----
    exclusion_radius = min(base_exclusion_radius, (T - 1) // 2)

    if exclusion_radius == 0:
        return torch.tensor(0.0, device=device)

    # time index per row
    time_idx = torch.arange(T, device=device).repeat(B)     # [BT]

    # batch index per row
    batch_idx = torch.arange(B, device=device).repeat_interleave(T)

    # same batch & temporally close → mask out
    temporal_dist = torch.abs(time_idx[:, None] - time_idx[None, :])
    same_batch = batch_idx[:, None] == batch_idx[None, :]

    invalid_negatives = same_batch & (temporal_dist <= exclusion_radius)

    # allow diagonal (positive pairs)
    diag = torch.eye(B*T, device=device, dtype=torch.bool)
    invalid_negatives = invalid_negatives & (~diag)

    # mask invalid negatives
    sim = sim.masked_fill(invalid_negatives, -1e9)

    # positives are diagonal
    labels = torch.arange(B*T, device=device)

    return F.cross_entropy(sim, labels)



def hierarchical_ts2vec_loss_v2(
    z1, z2,
    temperature=0.1,
    exclusion_radius=5,
    min_time=2
):
    total_loss = 0.0
    depth = 0

    while z1.size(1) >= min_time:
        T = z1.size(1)

        if T > 2 * exclusion_radius + 1:
            total_loss += ts2vec_contrastive_loss_vectorized(
                z1, z2,
                temperature,
                exclusion_radius
            )
            depth += 1

        z1 = temporal_pooling(z1)
        z2 = temporal_pooling(z2)

    return total_loss / max(depth, 1)



In [35]:
def temporal_contrast_score(Z, k=20):
    """
    This function computes the temporal contrast score for a given set of embeddings Z.
    Z: [T, C] numpy array of embeddings
    k: temporal gap for negative pairs
    """
    pos = []
    neg = []
    
    for i in range(len(Z) - k - 1):
        pos.append(np.dot(Z[i], Z[i+1]))
        neg.append(np.dot(Z[i], Z[i+k]))
    
    return np.mean(pos) - np.mean(neg)

# score = temporal_contrast_score(embeddings_full)
def temporal_contrast_score_multi(Z, pos_gap=1, neg_gaps=(10,20,40)):
    pos = []
    neg = []

    for i in range(len(Z) - max(neg_gaps) - 1):
        pos.append(np.dot(Z[i], Z[i+pos_gap]))
        for k in neg_gaps:
            neg.append(np.dot(Z[i], Z[i+k]))

    return np.mean(pos) - np.mean(neg)


from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

def predictive_score(Z, forward_returns):
    X = Z[:-1]
    y = forward_returns[1:]

    model = Ridge(alpha=1.0)
    model.fit(X, y)
    y_pred = model.predict(X)

    return r2_score(y, y_pred)

In [36]:
from dataclasses import dataclass

@dataclass
class WindowParams:
    window_size: int

@dataclass
class ArchitectureParams:
    hidden_dim: int
    num_layers: int

@dataclass
class ContrastiveParams:
    temperature: float
    exclusion_radius: int

@dataclass
class AugmentationParams:
    mask_ratio: float
    jitter_sigma: float

@dataclass
class OptimizationParams:
    learning_rate: float
    batch_size: int

In [37]:

long={'window': {'window_size': 180},
 'architecture': {'hidden_dim': 256, 'num_layers': 4},
 'contrastive': {'temperature': 0.10901392823581604, 'exclusion_radius': 18},
 'augmentation': {'mask_ratio': 0.10112014206194599,
  'jitter_sigma': 0.019884546877356246},
 'optimization': {'learning_rate': 0.0012543308801674451, 'batch_size': 64}}


short = {'window': 60,
 'architecture': {'hidden_dim': 128, 'num_layers': 4},
 'contrastive': {'temperature': 0.09418251159949012, 'exclusion_radius': 3},
 'augmentation': {'mask_ratio': 0.1153883102823196,
  'jitter_sigma': 0.006324426296094188},
 'optimization': {'learning_rate': 0.0017009936483370917, 'batch_size': 32}}



long_term_model_hyperparams_config = {
    "window_size": [30, 180, 30],  # range for window size  
    "hidden_dim": [ 128, 256,512],    
    "num_layers": [3, 10],
    "temperature": [0.05, 0.5],
    "exclusion_radius": [2, 20],
    "mask_ratio": [0.1, 0.5],
    "jitter_sigma": [0.005, 0.05],
    "learning_rate": [1e-4, 3e-3],
    "batch_size": [32,64,128]
}

short_term_model_hyperparams_config = {
    "window_size": [40, 80, 20],  # range for window size
    "hidden_dim": [32, 64, 128],
    "num_layers": [2, 7],
    "temperature": [0.05, 0.5],
    "exclusion_radius": [1, 5],
    "mask_ratio": [0.1, 0.5],
    "jitter_sigma": [0.005, 0.05],
    "learning_rate": [1e-4, 3e-3],
    "batch_size": [16,32,64]
}

mode = "short_term"  # or "short_term"

if mode == "long_term":
    hyperparams_config = long_term_model_hyperparams_config
else:
    hyperparams_config = short_term_model_hyperparams_config

In [46]:
import optuna
import torch
import numpy as np

class BaseTS2VecObjective:
    def __init__(self, data, device, fixed_params):
        self.data = data
        self.device = device
        self.fixed_params = fixed_params  # dict of params from previous stages

    def build_model(self, arch_params):
        return TS2VecModel(
            input_dim=self.data.shape[1],
            hidden_dim=arch_params.hidden_dim,
            num_layers=arch_params.num_layers
        ).to(self.device)

    def compute_score(self, model):
        model.eval()
        with torch.no_grad():
            full_tensor = torch.tensor(self.data.values, dtype=torch.float32).unsqueeze(0).to(self.device)
            z = model(full_tensor).squeeze(0).cpu().numpy()
            # use forward returns for predictive score
            return predictive_score(z, log_return.values)

    def train_model(self, model, train_loader, contrastive_params, aug_params, opt_params, epochs=3):
        optimizer = torch.optim.Adam(model.parameters(), lr=opt_params.learning_rate)

        model.train()
        for _ in range(epochs):  # small epochs for HPO
            for x in train_loader:
                x = x.to(self.device)
                x1 = jitter(time_mask(x, aug_params.mask_ratio), aug_params.jitter_sigma)
                x2 = jitter(time_mask(x, aug_params.mask_ratio), aug_params.jitter_sigma)

                z1 = model(x1)
                z2 = model(x2)

                loss = hierarchical_ts2vec_loss_v2(
                    z1, z2,
                    temperature=contrastive_params.temperature,
                    exclusion_radius=contrastive_params.exclusion_radius
                )

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        return model
    


In [47]:
def create_loader(data, window_size, batch_size):
    dataset = TimeSeriesWindowDataset(data.values, window_size)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)


In [48]:

class WindowStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):
        window_size = trial.suggest_int("window_size", hyperparams_config["window_size"][0] , hyperparams_config["window_size"][1], step=hyperparams_config["window_size"][2])

        window_params = WindowParams(window_size)

        train_loader = create_loader(self.data, window_size, batch_size=64)

        # fixed architecture for stage 0
        arch_params = ArchitectureParams(hidden_dim=128, num_layers=5)
        contrastive_params = ContrastiveParams(0.1, 5)
        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("window_params", window_params.__dict__)

        return score

In [49]:
type(data.values)

numpy.ndarray

In [50]:
study_window = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="window_stage",
    load_if_exists=True
)

# suggest optimal trail count based on combination of parameters 


study_window.optimize(WindowStageObjective(data, device, {}), n_trials=7)

[I 2026-02-27 02:59:12,359] A new study created in RDB with name: window_stage
[I 2026-02-27 02:59:15,110] Trial 0 finished with value: 0.047421710637158054 and parameters: {'window_size': 60}. Best is trial 0 with value: 0.047421710637158054.
[I 2026-02-27 02:59:16,841] Trial 1 finished with value: 0.046483482350964 and parameters: {'window_size': 40}. Best is trial 0 with value: 0.047421710637158054.
[I 2026-02-27 02:59:18,895] Trial 2 finished with value: 0.05042184240229475 and parameters: {'window_size': 60}. Best is trial 2 with value: 0.05042184240229475.
[I 2026-02-27 02:59:20,946] Trial 3 finished with value: 0.050324870227630214 and parameters: {'window_size': 60}. Best is trial 2 with value: 0.05042184240229475.
[I 2026-02-27 02:59:22,361] Trial 4 finished with value: 0.053487711051270814 and parameters: {'window_size': 40}. Best is trial 4 with value: 0.053487711051270814.
[I 2026-02-27 02:59:24,417] Trial 5 finished with value: 0.04495540433129652 and parameters: {'window_

In [ ]:
class ArchitectureStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):

        hidden_dim = trial.suggest_categorical("hidden_dim", hyperparams_config["hidden_dim"])
        num_layers = trial.suggest_int("num_layers", hyperparams_config["num_layers"][0], hyperparams_config["num_layers"][1])

        arch_params = ArchitectureParams(hidden_dim, num_layers)

        window_size = self.fixed_params["window_size"]

        train_loader = create_loader(self.data, window_size, batch_size=64)

        contrastive_params = ContrastiveParams(0.1, 5)
        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("arch_params", arch_params.__dict__)

        return score
    
study_arch = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="arch_stage",
    load_if_exists=True
)

n_trials = 30

study_arch.optimize(ArchitectureStageObjective(data, device, {**study_window.best_trial.user_attrs["window_params"]} ), n_trials= 30)

[I 2026-02-27 03:02:15,977] Using an existing study with name 'arch_stage' instead of creating a new one.


[W 2026-02-27 03:02:15,999] Trial 32 failed with parameters: {} because of the following error: ValueError('CategoricalDistribution does not support dynamic value space.').
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.12/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/jf/1r7jb1_s6ys1p0n0n56s2qsh0000gn/T/ipykernel_13531/82425197.py", line 5, in __call__
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 128, 256, 512])
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/lib/python3.12/site-packages/optuna/trial/_trial.py", line 402, in suggest_categorical
    return self._suggest(name, CategoricalDistribution(choices=choices))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/lib/python3.12/site-packages/optuna/trial/_trial.py", line 637, in 

ValueError: CategoricalDistribution does not support dynamic value space.

In [ ]:
# Contrastive stage:

# temperature = trial.suggest_float("temperature", 0.05, 0.5, log=True)
# exclusion_radius = trial.suggest_int("exclusion_radius", 2, 20)

class ContrastiveStageObjective(BaseTS2VecObjective):

    def __call__(self, trial):

        temperature = trial.suggest_float("temperature", hyperparams_config["temperature"][0], hyperparams_config["temperature"][1], log=True)
        exclusion_radius = trial.suggest_int("exclusion_radius", hyperparams_config["exclusion_radius"][0], hyperparams_config["exclusion_radius"][1])

        contrastive_params = ContrastiveParams(temperature, exclusion_radius)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])

        train_loader = create_loader(self.data, window_size, batch_size=64)

        aug_params = AugmentationParams(0.2, 0.02)
        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("contrastive_params", contrastive_params.__dict__)

        return score
    
study_contrastive = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="contrastive_stage",
    load_if_exists=True
)

study_contrastive.optimize(ContrastiveStageObjective(data, device, 
                                                     {
    **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"]
}
), n_trials=50)

[I 2026-02-23 12:58:56,162] Using an existing study with name 'contrastive_stage' instead of creating a new one.
[I 2026-02-23 12:58:58,077] Trial 0 finished with value: 0.6793023943901062 and parameters: {'temperature': 0.42552969467311136, 'exclusion_radius': 14}. Best is trial 0 with value: 0.6793023943901062.
[I 2026-02-23 12:58:59,947] Trial 1 finished with value: 0.7771525382995605 and parameters: {'temperature': 0.08357242993131461, 'exclusion_radius': 12}. Best is trial 1 with value: 0.7771525382995605.
[I 2026-02-23 12:59:01,567] Trial 2 finished with value: 0.7561784386634827 and parameters: {'temperature': 0.131630796345488, 'exclusion_radius': 18}. Best is trial 1 with value: 0.7771525382995605.
[I 2026-02-23 12:59:03,187] Trial 3 finished with value: 0.7088033556938171 and parameters: {'temperature': 0.3456678373263851, 'exclusion_radius': 18}. Best is trial 1 with value: 0.7771525382995605.
[I 2026-02-23 12:59:04,811] Trial 4 finished with value: 0.6718445420265198 and pa

In [ ]:
# Augmentation stage:

# mask_ratio = trial.suggest_float("mask_ratio", 0.1, 0.4)
# jitter_sigma = trial.suggest_float("jitter_sigma", 0.005, 0.05, log=True)

class AugmentationStageObjective(BaseTS2VecObjective):
    
    def __call__(self, trial):

        mask_ratio = trial.suggest_float("mask_ratio", hyperparams_config["mask_ratio"][0], hyperparams_config["mask_ratio"][1])
        jitter_sigma = trial.suggest_float("jitter_sigma", hyperparams_config["jitter_sigma"][0], hyperparams_config["jitter_sigma"][1], log=True)

        aug_params = AugmentationParams(mask_ratio, jitter_sigma)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])
        contrastive_params = ContrastiveParams(self.fixed_params["temperature"], self.fixed_params["exclusion_radius"])

        train_loader = create_loader(self.data, window_size, batch_size=64)

        opt_params = OptimizationParams(1e-3, 64)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("aug_params", aug_params.__dict__)

        return score
    
study_aug = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="augmentation_stage",
    load_if_exists=True
)
study_aug.optimize(AugmentationStageObjective(data, device, 
                                                     {
      **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"],
    **study_contrastive.best_trial.user_attrs["contrastive_params"]
}
), n_trials=50)

[I 2026-02-23 14:09:11,185] Using an existing study with name 'augmentation_stage' instead of creating a new one.


[I 2026-02-23 14:09:13,194] Trial 8 finished with value: 0.8583179712295532 and parameters: {'mask_ratio': 0.12665154206769594, 'jitter_sigma': 0.006066175263652591}. Best is trial 4 with value: 0.8638263940811157.
[I 2026-02-23 14:09:15,238] Trial 9 finished with value: 0.7540524005889893 and parameters: {'mask_ratio': 0.3044021721555483, 'jitter_sigma': 0.00809025315220967}. Best is trial 4 with value: 0.8638263940811157.
[I 2026-02-23 14:09:17,212] Trial 10 finished with value: 0.869362473487854 and parameters: {'mask_ratio': 0.10893971080565641, 'jitter_sigma': 0.046956947754562745}. Best is trial 10 with value: 0.869362473487854.
[I 2026-02-23 14:09:19,195] Trial 11 finished with value: 0.6780766248703003 and parameters: {'mask_ratio': 0.39139443299241805, 'jitter_sigma': 0.04933521504431264}. Best is trial 10 with value: 0.869362473487854.
[I 2026-02-23 14:09:21,171] Trial 12 finished with value: 0.8658281564712524 and parameters: {'mask_ratio': 0.10212459240894632, 'jitter_sigma

In [ ]:
# Optimization stage:

# learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)
# batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

class OptimizationStageObjective(BaseTS2VecObjective):
    
    def __call__(self, trial):

        learning_rate = trial.suggest_float("learning_rate", hyperparams_config["learning_rate"][0], hyperparams_config["learning_rate"][1], log=True)
        batch_size = trial.suggest_categorical("batch_size", hyperparams_config["batch_size"])

        opt_params = OptimizationParams(learning_rate, batch_size)

        window_size = self.fixed_params["window_size"]
        arch_params = ArchitectureParams(self.fixed_params["hidden_dim"], self.fixed_params["num_layers"])
        contrastive_params = ContrastiveParams(self.fixed_params["temperature"], self.fixed_params["exclusion_radius"])
        aug_params = AugmentationParams(self.fixed_params["mask_ratio"], self.fixed_params["jitter_sigma"])

        train_loader = create_loader(self.data, window_size, batch_size)

        model = self.build_model(arch_params)

        self.train_model(model, train_loader, contrastive_params, aug_params, opt_params)

        score = self.compute_score(model)

        trial.set_user_attr("opt_params", opt_params.__dict__)

        return score
    
study_opt = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    storage="sqlite:///ts2vec_hpo.db",
    study_name="optimization_stage",
    load_if_exists=True
)
study_opt.optimize(OptimizationStageObjective(data, device,
                                                     {
  **study_window.best_trial.user_attrs["window_params"],
    **study_arch.best_trial.user_attrs["arch_params"],
    **study_contrastive.best_trial.user_attrs["contrastive_params"],
    **study_aug.best_trial.user_attrs["aug_params"]
}), n_trials=50)

[I 2026-02-23 14:22:07,500] A new study created in RDB with name: optimization_stage


[I 2026-02-23 14:22:09,552] Trial 0 finished with value: 0.8860816955566406 and parameters: {'learning_rate': 0.0024841809279477173, 'batch_size': 64}. Best is trial 0 with value: 0.8860816955566406.
[I 2026-02-23 14:22:12,321] Trial 1 finished with value: 0.8970596194267273 and parameters: {'learning_rate': 0.0016340155948858838, 'batch_size': 32}. Best is trial 1 with value: 0.8970596194267273.
[I 2026-02-23 14:22:14,283] Trial 2 finished with value: 0.7487084865570068 and parameters: {'learning_rate': 0.00011630511430376669, 'batch_size': 64}. Best is trial 1 with value: 0.8970596194267273.
[I 2026-02-23 14:22:16,235] Trial 3 finished with value: 0.8096197247505188 and parameters: {'learning_rate': 0.00031167405254851053, 'batch_size': 64}. Best is trial 1 with value: 0.8970596194267273.
[I 2026-02-23 14:22:18,196] Trial 4 finished with value: 0.7981594204902649 and parameters: {'learning_rate': 0.0002786714268131344, 'batch_size': 64}. Best is trial 1 with value: 0.8970596194267273

In [ ]:
from datetime import datetime

class TS2VecHyperparameterPipeline:
    def __init__(self, data, device, mode="long_term"):
        self.data = data
        self.device = device
        self.results = {}
        self.mode = mode  # "long_term" or "short_term"

    def _run_stage(self, name, objective_cls, fixed_params, n_trials, result_key, attr_key):
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(),
            storage="sqlite:///ts2vec_hpo.db",
            study_name=f"{name}_stage_{self.mode}_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
            load_if_exists=True,
        )
        study.optimize(objective_cls(self.data, self.device, fixed_params), n_trials=n_trials)
        self.results[result_key] = study.best_trial.user_attrs[attr_key]

    def run(self):
        self._run_stage(
            "window",
            WindowStageObjective,
            {},
            n_trials=7,
            result_key="window",
            attr_key="window_params",
        )
        self._run_stage(
            "arch",
            ArchitectureStageObjective,
            self.results["window"],
            n_trials=30,
            result_key="architecture",
            attr_key="arch_params",
        )
        self._run_stage(
            "contrastive",
            ContrastiveStageObjective,
            {**self.results["window"], **self.results["architecture"]},
            n_trials=50,
            result_key="contrastive",
            attr_key="contrastive_params",
        )
        self._run_stage(
            "augmentation",
            AugmentationStageObjective,
            {**self.results["window"], **self.results["architecture"], **self.results["contrastive"]},
            n_trials=50,
            result_key="augmentation",
            attr_key="aug_params",
        )
        self._run_stage(
            "optimization",
            OptimizationStageObjective,
            {
                **self.results["window"],
                **self.results["architecture"],
                **self.results["contrastive"],
                **self.results["augmentation"],
            },
            n_trials=50,
            result_key="optimization",
            attr_key="opt_params",
        )
        return self.results

# Run the pipeline
pipeline = TS2VecHyperparameterPipeline(data, device, mode="long_term")
final_results = pipeline.run()
print(final_results)

In [ ]:
# train final model with best hyperparameters

best_model_params = {
    **final_results["window"],
    **final_results["architecture"],
    **final_results["contrastive"],
    **final_results["augmentation"],
    **final_results["optimization"]
}



In [33]:
# list all the best parameters from each stage
param_summary = {
    "window": 60,
    "architecture": study_arch.best_trial.user_attrs["arch_params"],
    "contrastive": study_contrastive.best_trial.user_attrs["contrastive_params"],
    "augmentation": study_aug.best_trial.user_attrs["aug_params"],
    "optimization": study_opt.best_trial.user_attrs["opt_params"]
}

```json
{'window': {'window_size': 180},
 'architecture': {'hidden_dim': 256, 'num_layers': 4},
 'contrastive': {'temperature': 0.10901392823581604, 'exclusion_radius': 18},
 'augmentation': {'mask_ratio': 0.10112014206194599,
  'jitter_sigma': 0.019884546877356246},
 'optimization': {'learning_rate': 0.0012543308801674451, 'batch_size': 64}}
```
```json
{'window': 60,
 'architecture': {'hidden_dim': 128, 'num_layers': 4},
 'contrastive': {'temperature': 0.09418251159949012, 'exclusion_radius': 3},
 'augmentation': {'mask_ratio': 0.1153883102823196,
  'jitter_sigma': 0.006324426296094188},
 'optimization': {'learning_rate': 0.0017009936483370917, 'batch_size': 32}}
```
 

In [34]:
param_summary

{'window': 60,
 'architecture': {'hidden_dim': 128, 'num_layers': 4},
 'contrastive': {'temperature': 0.09418251159949012, 'exclusion_radius': 3},
 'augmentation': {'mask_ratio': 0.1153883102823196,
  'jitter_sigma': 0.006324426296094188},
 'optimization': {'learning_rate': 0.0017009936483370917, 'batch_size': 32}}